In [ ]:
# ==============================================================================
# 🚀 OmniVoice GPU Inference Server trên Kaggle (30 Giờ GPU T4 Miễn Phí/Tuần)
# ==============================================================================

# 1. Cấu hình Ngrok (Điền mã authtoken và static domain nếu có)
NGROK_AUTHTOKEN = ""  # Nhập mã Ngrok Authtoken của bạn vào đây
NGROK_STATIC_DOMAIN = ""  # Nhập tên miền tĩnh Ngrok (nếu có)

import os
import sys
import time
import subprocess
import urllib.request
import json
import base64

print("🔍 [1/4] Đang kiểm tra GPU...")
gpu_check = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
if gpu_check.returncode != 0:
    raise RuntimeError("❌ Chưa bật GPU! Vui lòng nhìn sang góc phải màn hình Kaggle: Session options -> Accelerator -> Chọn GPU T4 x2 rồi bấm Turn on.")
print("✅ Nhận diện GPU thành công!\n" + gpu_check.stdout.split("\n")[8])

print("\n📦 [2/4] Đang cài đặt thư viện phụ thuộc...")
reqs = [
    "omnivoice", "accelerate>=0.34.0", "webdataset", "safetensors", "transformers>=4.40.0",
    "soundfile", "librosa", "pydub", "scipy", "faster-whisper", "fastapi", "uvicorn[standard]",
    "python-multipart", "httpx", "pyngrok"
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + reqs, check=True)
print("✅ Đã cài đặt xong toàn bộ thư viện phụ thuộc!")

print("\n⚙️ [3/4] Đang thiết lập worker server...")
WORKER_B64 = "..."
with open("colab_worker.py", "wb") as f:
    f.write(base64.b64decode(WORKER_B64))

log_file = open("worker.log", "w", encoding="utf-8")
server_proc = subprocess.Popen([sys.executable, "-u", "colab_worker.py"], stdout=log_file, stderr=subprocess.STDOUT)

is_ready = False
print("⏳ Đang tải trọng số OmniVoice vào VRAM (vui lòng đợi ~30 giây)...", end="", flush=True)
for _ in range(45):
    if server_proc.poll() is not None:
        break
    try:
        with urllib.request.urlopen("http://127.0.0.1:8000/api/remote/health", timeout=2) as resp:
            if resp.status == 200:
                data = json.loads(resp.read().decode())
                is_ready = True
                print(f"\n✅ Server đã sẵn sàng! [{data.get('gpu_name')} - {data.get('vram_total_gb')}GB VRAM]")
                break
    except Exception:
        pass
    print(".", end="", flush=True)
    time.sleep(2)

if not is_ready:
    log_file.close()
    with open("worker.log", "r", encoding="utf-8") as f:
        err_output = f.read()
    raise RuntimeError(f"❌ Server không khởi động được! Log lỗi:\n\n{err_output}")

print("\n🌐 [4/4] Đang mở đường hầm kết nối Ngrok...")
if NGROK_AUTHTOKEN.strip():
    from pyngrok import ngrok
    ngrok.set_auth_token(NGROK_AUTHTOKEN.strip())
    domain_val = NGROK_STATIC_DOMAIN.strip() if NGROK_STATIC_DOMAIN.strip() else None
    if domain_val:
        tunnel = ngrok.connect(8000, "http", domain=domain_val)
    else:
        tunnel = ngrok.connect(8000, "http")
    tunnel_url = tunnel.public_url
    print(f"\n🎉 KẾT NỐI NGROK THÀNH CÔNG!\n🔗 Link GPU Cloud: {tunnel_url}")
    print("\n=======================================================================")
    print("👉 HÃY DÁN LINK NÀY VÀO FILE 'backend/.env' HOẶC TRANG CÀI ĐẶT TRÊN WEB:")
    print(f"   REMOTE_GPU_URL={tunnel_url}")
    print("=======================================================================")
else:
    print("⚠️ Bạn chưa điền NGROK_AUTHTOKEN ở đầu ô code!")


